<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/codes/MNIST_1D_PCA_4_ZZ_QKE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ZZ QKE


In [ ]:
!pip install qiskit qiskit_machine_learning qiskit-algorithms


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00


In [ ]:
!pip install mnist1d --break-system-packages

In [ ]:


import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
import mnist1d

# ----------------------------- Config ------------------------------------
N_DIM = 4
N_TRAIN = 250
N_TEST = 250
N_REPS = 30
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
CV_FOLDS = 5
DIGITS = (3, 5)

FEATURE_MAP = ZZFeatureMap(feature_dimension=N_DIM, reps=2)
KERNEL = FidelityStatevectorKernel(feature_map=FEATURE_MAP)


# ------------------------- Data loading -----------------------------------
def load_mnist1d_binary(digits=DIGITS):
    """Load MNIST-1D and keep only the two requested classes."""
    args = mnist1d.data.get_dataset_args()
    data = mnist1d.data.make_dataset(args)
    X = np.concatenate([data["x"], data["x_test"]])
    y = np.concatenate([data["y"], data["y_test"]])
    mask = np.isin(y, digits)
    X, y = X[mask], y[mask]
    y = np.where(y == digits[0], -1, 1)
    return X, y


def scaled_kernel(X1, X2, lam):
    """Fidelity kernel matrix with angle vectors pre-scaled by bandwidth lam."""
    return KERNEL.evaluate(x_vec=X1 * lam, y_vec=X2 * lam)


# ------------------------- Experiment loop ---------------------------------
def run_experiment(X_raw, y, n_reps=N_REPS, verbose=True):
    results = {
        "train": {"acc": [], "kappa": [], "f1": []},
        "test": {"acc": [], "kappa": [], "f1": []},
    }

    for rep in range(n_reps):
        X_tr_full, X_te_full, y_tr, y_te = train_test_split(
            X_raw, y,
            train_size=N_TRAIN, test_size=N_TEST,
            stratify=y, random_state=rep,
        )

        # PCA fit on train only, applied to both splits
        pca = PCA(n_components=N_DIM, random_state=rep)
        X_tr_pca = pca.fit_transform(X_tr_full)
        X_te_pca = pca.transform(X_te_full)

        # Angle-encoding scale: [0, 2*pi], parameters learned from train only
        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_tr = scaler.fit_transform(X_tr_pca)
        X_te = scaler.transform(X_te_pca)

        # ---- Grid search over (lambda, C) via 5-fold CV on train ----
        best_score, best_C, best_lam = -1.0, None, None
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rep)

        for lam in LAMBDA_GRID:
            K_full = scaled_kernel(X_tr, X_tr, lam)
            for C in C_GRID:
                fold_acc = []
                for tr_idx, val_idx in skf.split(X_tr, y_tr):
                    K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                    K_val = K_full[np.ix_(val_idx, tr_idx)]
                    clf = SVC(kernel="precomputed", C=C)
                    clf.fit(K_tr, y_tr[tr_idx])
                    pred = clf.predict(K_val)
                    fold_acc.append(accuracy_score(y_tr[val_idx], pred))
                mean_acc = float(np.mean(fold_acc))
                if mean_acc > best_score:
                    best_score, best_C, best_lam = mean_acc, C, lam

        # ---- Refit on full train split with the winning hyperparams ----
        K_train = scaled_kernel(X_tr, X_tr, best_lam)
        K_test = scaled_kernel(X_te, X_tr, best_lam)

        clf = SVC(kernel="precomputed", C=best_C)
        clf.fit(K_train, y_tr)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        results["train"]["acc"].append(accuracy_score(y_tr, pred_train))
        results["train"]["kappa"].append(cohen_kappa_score(y_tr, pred_train))
        results["train"]["f1"].append(f1_score(y_tr, pred_train, average="macro"))

        results["test"]["acc"].append(accuracy_score(y_te, pred_test))
        results["test"]["kappa"].append(cohen_kappa_score(y_te, pred_test))
        results["test"]["f1"].append(f1_score(y_te, pred_test, average="macro"))

        if verbose:
            print(f"[rep {rep + 1:2d}/{n_reps}] best_C={best_C:<6} "
                  f"best_lambda={best_lam:<6} "
                  f"test_acc={results['test']['acc'][-1]:.3f}")

    return results


def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"


def print_table2_row(results, dataset="MNIST-1D-PCA-4", mapping="ZZFeatureMap"):
    print(f"\n=== Table 2 style summary: {dataset} | qSVM_ZZ (plain QKE, n={len(results['test']['acc'])}) ===")
    header = f"{'Dataset':<18}{'Map':<14}{'Params':<10}{'Metric':<16}{'Train':<18}{'Test':<18}"
    print(header)
    print("-" * len(header))
    for metric, label in [("acc", "Accuracy"), ("kappa", "Cohen's kappa"), ("f1", "Macro F1")]:
        print(f"{dataset:<18}{mapping:<14}{'-':<10}{label:<16}"
              f"{fmt(results['train'][metric]):<18}{fmt(results['test'][metric]):<18}")


if __name__ == "__main__":
    X_raw, y = load_mnist1d_binary()
    print(f"Loaded MNIST-1D digits {DIGITS}: {X_raw.shape[0]} samples, "
          f"raw dim={X_raw.shape[1]}")

    results = run_experiment(X_raw, y, n_reps=N_REPS)
    print_table2_row(results)

/tmp/ipykernel_2682/3656747228.py:58: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  FEATURE_MAP = ZZFeatureMap(feature_dimension=N_DIM, reps=2)


Loaded MNIST-1D digits (3, 5): 1000 samples, raw dim=40
[rep  1/30] best_C=100    best_lambda=0.01   test_acc=0.924
[rep  2/30] best_C=100    best_lambda=0.01   test_acc=0.912
[rep  3/30] best_C=100    best_lambda=0.01   test_acc=0.828
[rep  4/30] best_C=100    best_lambda=0.01   test_acc=0.872
[rep  5/30] best_C=1      best_lambda=0.1    test_acc=0.848
[rep  6/30] best_C=100    best_lambda=0.01   test_acc=0.888
[rep  7/30] best_C=1      best_lambda=0.1    test_acc=0.864
[rep  8/30] best_C=1      best_lambda=0.1    test_acc=0.864
[rep  9/30] best_C=100    best_lambda=0.01   test_acc=0.872
[rep 10/30] best_C=100    best_lambda=0.01   test_acc=0.872
[rep 11/30] best_C=100    best_lambda=0.01   test_acc=0.872
[rep 12/30] best_C=10     best_lambda=0.01   test_acc=0.852
[rep 13/30] best_C=1      best_lambda=0.1    test_acc=0.832
[rep 14/30] best_C=100    best_lambda=0.01   test_acc=0.880
[rep 15/30] best_C=1      best_lambda=0.1    test_acc=0.876
[rep 16/30] best_C=1      best_lambda=0.1   

QKT Shared 3


In [ ]:


import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import TrainableFidelityStatevectorKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer
from qiskit_machine_learning.utils.loss_functions import SVCLoss
from qiskit_algorithms.optimizers import SPSA
import mnist1d

# ----------------------------- Config ------------------------------------
N_DIM = 4
N_TRAIN = 250
N_TEST = 250
N_REPS = 10
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
CV_FOLDS = 5
DIGITS = (3, 5)

PARAMETERIZATION = "shared"   # 3 trainable parameters, shared across all qubits
QKT_MAXITER = 100
QKT_C = 1.0                   # C used inside the SVCLoss (weighted alignment) itself


# ----------------------- Trainable circuit build ---------------------------
def build_trainable_zz_circuit(n_qubits, parameterization="shared"):
    """
    Build Uθ(x) = U_ZZ(x) · Vθ, i.e. Vθ|0> prepared first, ZZFeatureMap(x)
    applied on top (Eq. 19). Returns (circuit, theta_params).
    """
    if parameterization == "shared":
        theta = ParameterVector("theta", 3)
        angle_sets = [theta] * n_qubits
    elif parameterization == "dedicated":
        theta = ParameterVector("theta", 3 * n_qubits)
        angle_sets = [theta[3 * q: 3 * q + 3] for q in range(n_qubits)]
    else:
        raise ValueError("parameterization must be 'shared' or 'dedicated'")

    qc = QuantumCircuit(n_qubits)
    for q in range(n_qubits):
        t1, t2, t3 = angle_sets[q]
        # RXYZ(t1,t2,t3) [Eq. 22] == Qiskit U(theta=t1, phi=t3, lambda=t2)
        qc.u(t1, t3, t2, q)

    feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)
    qc.compose(feature_map, qubits=range(n_qubits), inplace=True)
    return qc, list(theta)


# ------------------------- Data loading -----------------------------------
def load_mnist1d_binary(digits=DIGITS):
    args = mnist1d.data.get_dataset_args()
    data = mnist1d.data.make_dataset(args)
    X = np.concatenate([data["x"], data["x_test"]])
    y = np.concatenate([data["y"], data["y_test"]])
    mask = np.isin(y, digits)
    X, y = X[mask], y[mask]
    y = np.where(y == digits[0], -1, 1)
    return X, y


# ------------------------- Experiment loop ---------------------------------
def run_experiment(X_raw, y, n_reps=N_REPS, verbose=True):
    results = {
        "train": {"acc": [], "kappa": [], "f1": []},
        "test": {"acc": [], "kappa": [], "f1": []},
    }
    n_params_used = None

    for rep in range(n_reps):
        X_tr_full, X_te_full, y_tr, y_te = train_test_split(
            X_raw, y,
            train_size=N_TRAIN, test_size=N_TEST,
            stratify=y, random_state=rep,
        )

        pca = PCA(n_components=N_DIM, random_state=rep)
        X_tr_pca = pca.fit_transform(X_tr_full)
        X_te_pca = pca.transform(X_te_full)

        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_tr = scaler.fit_transform(X_tr_pca)
        X_te = scaler.transform(X_te_pca)

        # ---- Fresh trainable circuit + kernel for this repetition ----
        circuit, theta_params = build_trainable_zz_circuit(N_DIM, PARAMETERIZATION)
        n_params_used = len(theta_params)
        base_kernel = TrainableFidelityStatevectorKernel(
            feature_map=circuit, training_parameters=theta_params
        )

        # ---- QKT: maximize weighted kernel-target alignment on train ----
        optimizer = SPSA(maxiter=QKT_MAXITER, second_order=True)
        trainer = QuantumKernelTrainer(
            quantum_kernel=base_kernel,
            loss=SVCLoss(C=QKT_C),
            optimizer=optimizer,
            initial_point=[0.0] * n_params_used,   # theta0 = 0 (Sec. III-C)
        )
        qkt_result = trainer.fit(X_tr, y_tr)
        trained_kernel = qkt_result.quantum_kernel  # theta fixed at theta_opt

        # ---- Grid search over (C, lambda) via 5-fold CV, now with the
        #      *trained* kernel; lambda still rescales the angle-encoded
        #      data vector prior to feature-map evaluation ----
        best_score, best_C, best_lam = -1.0, None, None
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rep)

        for lam in LAMBDA_GRID:
            K_full = trained_kernel.evaluate(x_vec=X_tr * lam, y_vec=X_tr * lam)
            for C in C_GRID:
                fold_acc = []
                for tr_idx, val_idx in skf.split(X_tr, y_tr):
                    K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                    K_val = K_full[np.ix_(val_idx, tr_idx)]
                    clf = SVC(kernel="precomputed", C=C)
                    clf.fit(K_tr, y_tr[tr_idx])
                    pred = clf.predict(K_val)
                    fold_acc.append(accuracy_score(y_tr[val_idx], pred))
                mean_acc = float(np.mean(fold_acc))
                if mean_acc > best_score:
                    best_score, best_C, best_lam = mean_acc, C, lam

        # ---- Refit on full train split with the winning (C, lambda) ----
        K_train = trained_kernel.evaluate(x_vec=X_tr * best_lam, y_vec=X_tr * best_lam)
        K_test = trained_kernel.evaluate(x_vec=X_te * best_lam, y_vec=X_tr * best_lam)

        clf = SVC(kernel="precomputed", C=best_C)
        clf.fit(K_train, y_tr)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        results["train"]["acc"].append(accuracy_score(y_tr, pred_train))
        results["train"]["kappa"].append(cohen_kappa_score(y_tr, pred_train))
        results["train"]["f1"].append(f1_score(y_tr, pred_train, average="macro"))

        results["test"]["acc"].append(accuracy_score(y_te, pred_test))
        results["test"]["kappa"].append(cohen_kappa_score(y_te, pred_test))
        results["test"]["f1"].append(f1_score(y_te, pred_test, average="macro"))

        if verbose:
            print(f"[rep {rep + 1:2d}/{n_reps}] theta_opt={np.round(qkt_result.optimal_point, 3)} "
                  f"best_C={best_C:<6} best_lambda={best_lam:<6} "
                  f"test_acc={results['test']['acc'][-1]:.3f}")

    return results, n_params_used


def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"


def print_table2_row(results, n_params, dataset="MNIST-1D-PCA-4", mapping="ZZFeatureMap"):
    label_params = f"shared ({n_params})"
    print(f"\n=== Table 2 style summary: {dataset} | qSVM_ZZ_opt_s (n={len(results['test']['acc'])}) ===")
    header = f"{'Dataset':<18}{'Map':<14}{'Params':<16}{'Metric':<16}{'Train':<18}{'Test':<18}"
    print(header)
    print("-" * len(header))
    for metric, label in [("acc", "Accuracy"), ("kappa", "Cohen's kappa"), ("f1", "Macro F1")]:
        print(f"{dataset:<18}{mapping:<14}{label_params:<16}{label:<16}"
              f"{fmt(results['train'][metric]):<18}{fmt(results['test'][metric]):<18}")


if __name__ == "__main__":
    X_raw, y = load_mnist1d_binary()
    print(f"Loaded MNIST-1D digits {DIGITS}: {X_raw.shape[0]} samples, "
          f"raw dim={X_raw.shape[1]}")

    results, n_params = run_experiment(X_raw, y, n_reps=N_REPS)
    print_table2_row(results, n_params)

Loaded MNIST-1D digits (3, 5): 1000 samples, raw dim=40


/tmp/ipykernel_2682/2516129577.py:89: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep  1/10] theta_opt=[ 0.274  0.715 -0.55 ] best_C=100    best_lambda=0.01   test_acc=0.908


/tmp/ipykernel_2682/2516129577.py:89: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep  2/10] theta_opt=[ 0.513 -0.558  0.107] best_C=0.1    best_lambda=0.1    test_acc=0.868


...

[rep  8/10] theta_opt=[ 1.392  1.063 -0.148] best_C=1      best_lambda=0.1    test_acc=0.868


/tmp/ipykernel_2682/2516129577.py:89: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep  9/10] theta_opt=[ 0.282  0.085 -0.013] best_C=1      best_lambda=0.1    test_acc=0.872


/tmp/ipykernel_2682/2516129577.py:89: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep 10/10] theta_opt=[-0.226  0.097 -2.924] best_C=10     best_lambda=0.01   test_acc=0.848

=== Table 2 style summary: MNIST-1D-PCA-4 | qSVM_ZZ_opt_s (n=10) ===
Dataset           Map           Params          Metric          Train             Test              
----------------------------------------------------------------------------------------------------
MNIST-1D-PCA-4    ZZFeatureMap  shared (3)      Accuracy        0.898 (0.035)     0.866 (0.022)     
MNIST-1D-PCA-4    ZZFeatureMap  shared (3)      Cohen's kappa   0.796 (0.070)     0.731 (0.045)     
MNIST-1D-PCA-4    ZZFeatureMap  shared (3)      Macro F1        0.898 (0.035)     0.865 (0.022)     


ZZ QKT Dedicated 12


In [ ]:


import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import TrainableFidelityStatevectorKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer
from qiskit_machine_learning.utils.loss_functions import SVCLoss
from qiskit_algorithms.optimizers import SPSA
import mnist1d

# ----------------------------- Config ------------------------------------
N_DIM = 4
N_TRAIN = 250
N_TEST = 250
N_REPS = 10
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
CV_FOLDS = 5
DIGITS = (3, 5)

PARAMETERIZATION = "dedicated"   # 12 trainable parameters (3 per qubit, 4 qubits)
QKT_MAXITER = 100
QKT_C = 1.0                   # C used inside the SVCLoss (weighted alignment) itself


# ----------------------- Trainable circuit build ---------------------------
def build_trainable_zz_circuit(n_qubits, parameterization="shared"):
    """
    Build Uθ(x) = U_ZZ(x) · Vθ, i.e. Vθ|0> prepared first, ZZFeatureMap(x)
    applied on top (Eq. 19). Returns (circuit, theta_params).
    """
    if parameterization == "shared":
        theta = ParameterVector("theta", 3)
        angle_sets = [theta] * n_qubits
    elif parameterization == "dedicated":
        theta = ParameterVector("theta", 3 * n_qubits)
        angle_sets = [theta[3 * q: 3 * q + 3] for q in range(n_qubits)]
    else:
        raise ValueError("parameterization must be 'shared' or 'dedicated'")

    qc = QuantumCircuit(n_qubits)
    for q in range(n_qubits):
        t1, t2, t3 = angle_sets[q]
        # RXYZ(t1,t2,t3) [Eq. 22] == Qiskit U(theta=t1, phi=t3, lambda=t2)
        qc.u(t1, t3, t2, q)

    feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)
    qc.compose(feature_map, qubits=range(n_qubits), inplace=True)
    return qc, list(theta)


# ------------------------- Data loading -----------------------------------
def load_mnist1d_binary(digits=DIGITS):
    args = mnist1d.data.get_dataset_args()
    data = mnist1d.data.make_dataset(args)
    X = np.concatenate([data["x"], data["x_test"]])
    y = np.concatenate([data["y"], data["y_test"]])
    mask = np.isin(y, digits)
    X, y = X[mask], y[mask]
    y = np.where(y == digits[0], -1, 1)
    return X, y


# ------------------------- Experiment loop ---------------------------------
def run_experiment(X_raw, y, n_reps=N_REPS, verbose=True):
    results = {
        "train": {"acc": [], "kappa": [], "f1": []},
        "test": {"acc": [], "kappa": [], "f1": []},
    }
    n_params_used = None

    for rep in range(n_reps):
        X_tr_full, X_te_full, y_tr, y_te = train_test_split(
            X_raw, y,
            train_size=N_TRAIN, test_size=N_TEST,
            stratify=y, random_state=rep,
        )

        pca = PCA(n_components=N_DIM, random_state=rep)
        X_tr_pca = pca.fit_transform(X_tr_full)
        X_te_pca = pca.transform(X_te_full)

        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_tr = scaler.fit_transform(X_tr_pca)
        X_te = scaler.transform(X_te_pca)

        # ---- Fresh trainable circuit + kernel for this repetition ----
        circuit, theta_params = build_trainable_zz_circuit(N_DIM, PARAMETERIZATION)
        n_params_used = len(theta_params)
        base_kernel = TrainableFidelityStatevectorKernel(
            feature_map=circuit, training_parameters=theta_params
        )

        # ---- QKT: maximize weighted kernel-target alignment on train ----
        optimizer = SPSA(maxiter=QKT_MAXITER, second_order=True)
        trainer = QuantumKernelTrainer(
            quantum_kernel=base_kernel,
            loss=SVCLoss(C=QKT_C),
            optimizer=optimizer,
            initial_point=[0.0] * n_params_used,   # theta0 = 0 (Sec. III-C)
        )
        qkt_result = trainer.fit(X_tr, y_tr)
        trained_kernel = qkt_result.quantum_kernel  # theta fixed at theta_opt

        # ---- Grid search over (C, lambda) via 5-fold CV, now with the
        #      *trained* kernel; lambda still rescales the angle-encoded
        #      data vector prior to feature-map evaluation ----
        best_score, best_C, best_lam = -1.0, None, None
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rep)

        for lam in LAMBDA_GRID:
            K_full = trained_kernel.evaluate(x_vec=X_tr * lam, y_vec=X_tr * lam)
            for C in C_GRID:
                fold_acc = []
                for tr_idx, val_idx in skf.split(X_tr, y_tr):
                    K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                    K_val = K_full[np.ix_(val_idx, tr_idx)]
                    clf = SVC(kernel="precomputed", C=C)
                    clf.fit(K_tr, y_tr[tr_idx])
                    pred = clf.predict(K_val)
                    fold_acc.append(accuracy_score(y_tr[val_idx], pred))
                mean_acc = float(np.mean(fold_acc))
                if mean_acc > best_score:
                    best_score, best_C, best_lam = mean_acc, C, lam

        # ---- Refit on full train split with the winning (C, lambda) ----
        K_train = trained_kernel.evaluate(x_vec=X_tr * best_lam, y_vec=X_tr * best_lam)
        K_test = trained_kernel.evaluate(x_vec=X_te * best_lam, y_vec=X_tr * best_lam)

        clf = SVC(kernel="precomputed", C=best_C)
        clf.fit(K_train, y_tr)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        results["train"]["acc"].append(accuracy_score(y_tr, pred_train))
        results["train"]["kappa"].append(cohen_kappa_score(y_tr, pred_train))
        results["train"]["f1"].append(f1_score(y_tr, pred_train, average="macro"))

        results["test"]["acc"].append(accuracy_score(y_te, pred_test))
        results["test"]["kappa"].append(cohen_kappa_score(y_te, pred_test))
        results["test"]["f1"].append(f1_score(y_te, pred_test, average="macro"))

        if verbose:
            print(f"[rep {rep + 1:2d}/{n_reps}] theta_opt={np.round(qkt_result.optimal_point, 3)} "
                  f"best_C={best_C:<6} best_lambda={best_lam:<6} "
                  f"test_acc={results['test']['acc'][-1]:.3f}")

    return results, n_params_used


def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"


def print_table2_row(results, n_params, dataset="MNIST-1D-PCA-4", mapping="ZZFeatureMap"):
    label_params = f"dedicated ({n_params})"
    print(f"\n=== Table 2 style summary: {dataset} | qSVM_ZZ_opt_d (n={len(results['test']['acc'])}) ===")
    header = f"{'Dataset':<18}{'Map':<14}{'Params':<16}{'Metric':<16}{'Train':<18}{'Test':<18}"
    print(header)
    print("-" * len(header))
    for metric, label in [("acc", "Accuracy"), ("kappa", "Cohen's kappa"), ("f1", "Macro F1")]:
        print(f"{dataset:<18}{mapping:<14}{label_params:<16}{label:<16}"
              f"{fmt(results['train'][metric]):<18}{fmt(results['test'][metric]):<18}")


if __name__ == "__main__":
    X_raw, y = load_mnist1d_binary()
    print(f"Loaded MNIST-1D digits {DIGITS}: {X_raw.shape[0]} samples, "
          f"raw dim={X_raw.shape[1]}")

    results, n_params = run_experiment(X_raw, y, n_reps=N_REPS)
    print_table2_row(results, n_params)

Loaded MNIST-1D digits (3, 5): 1000 samples, raw dim=40


/tmp/ipykernel_2682/1699543248.py:100: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep  1/10] theta_opt=[ 1.305  0.718  0.035  0.579  0.383  0.518  0.632 -0.698  0.688  0.755
 -0.303 -0.26 ] best_C=100    best_lambda=0.01   test_acc=0.916


/tmp/ipykernel_2682/1699543248.py:100: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep  2/10] theta_opt=[ 0.601  0.008 -0.444  0.368 -0.063  0.471 -0.352 -0.634  0.11   0.863
  0.409 -0.155] best_C=100    best_lambda=0.01   test_acc=0.908


...

[rep  8/10] theta_opt=[ 0.741 -0.905 -0.372  0.561  0.235 -0.236  0.319 -1.141 -0.755  0.708
  0.579 -0.474] best_C=1      best_lambda=0.1    test_acc=0.844


/tmp/ipykernel_2682/1699543248.py:100: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep  9/10] theta_opt=[ 2.990e-01 -9.030e-01 -1.736e+00 -6.170e-01  4.340e-01  2.650e-01
  4.160e-01  9.530e-01 -1.000e-03 -5.610e-01  1.080e+00  1.025e+00] best_C=100    best_lambda=0.01   test_acc=0.864


/tmp/ipykernel_2682/1699543248.py:100: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)


[rep 10/10] theta_opt=[-0.432 -0.532 -0.163  0.29  -0.38   0.052 -0.585  0.068  0.348  0.987
 -0.354 -0.283] best_C=10     best_lambda=0.01   test_acc=0.848

=== Table 2 style summary: MNIST-1D-PCA-4 | qSVM_ZZ_opt_d (n=10) ===
Dataset           Map           Params          Metric          Train             Test              
----------------------------------------------------------------------------------------------------
MNIST-1D-PCA-4    ZZFeatureMap  dedicated (12)  Accuracy        0.890 (0.028)     0.873 (0.028)     
MNIST-1D-PCA-4    ZZFeatureMap  dedicated (12)  Cohen's kappa   0.779 (0.056)     0.746 (0.056)     
MNIST-1D-PCA-4    ZZFeatureMap  dedicated (12)  Macro F1        0.890 (0.028)     0.873 (0.028)     


Covariant QKT Dedicated 6

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
import mnist1d

# ----------------------------- Config ------------------------------------
N_DIM = 4
N_TRAIN = 250
N_TEST = 250
N_REPS = 30
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
CV_FOLDS = 5
DIGITS = (3, 5)

# --- QKT Settings ---
# Evaluates on the full training set as requested for rigorous benchmarking
QKT_MAXITER = 100

# ----------------------------- Feature Map --------------------------------
def CovariantFeatureMap_Dedicated6(feature_dimension):
    """
    Constructs a group-covariant feature map with a parameterized
    fiducial state using a 'Dedicated (6)' strategy.

    Since 6 parameters on 4 qubits is not perfectly symmetrical, we
    assign 1 full layer (4 params) + 1 partial layer (2 params).
    """
    x = ParameterVector('x', length=feature_dimension)
    theta = ParameterVector('θ', length=6) # Exactly 6 dedicated parameters
    qc = QuantumCircuit(feature_dimension, name="CovariantMap_QKT_Dedicated6")

    param_idx = 0

    # 1. Parameterized Fiducial State
    # --- Layer 1 (4 parameters) ---
    for i in range(feature_dimension):
        qc.ry(theta[param_idx], i)
        param_idx += 1

    # --- Entanglement ---
    if feature_dimension > 1:
        for i in range(feature_dimension - 1):
            qc.cx(i, i + 1)

    # --- Layer 2 (Partial: 2 parameters) ---
    # Apply to the first two qubits to use the remaining 2 parameters
    for i in range(2):
        qc.ry(theta[param_idx], i)
        param_idx += 1

    # 2. Covariant Data Encoding Layer (U(x) = exp(-i * x * Z))
    for i in range(feature_dimension):
        qc.rz(x[i], i)

    return qc, theta, x

PARAM_CIRCUIT, WEIGHT_PARAMS, DATA_PARAMS = CovariantFeatureMap_Dedicated6(feature_dimension=N_DIM)

# ------------------------- Helper Functions -------------------------------
def load_mnist1d_binary(digits=DIGITS):
    args = mnist1d.data.get_dataset_args()
    data = mnist1d.data.make_dataset(args)
    X = np.concatenate([data["x"], data["x_test"]])
    y = np.concatenate([data["y"], data["y_test"]])
    mask = np.isin(y, digits)
    X, y = X[mask], y[mask]
    y = np.where(y == digits[0], -1, 1)
    return X, y

def get_bound_kernel(theta_vals):
    param_dict = dict(zip(WEIGHT_PARAMS, theta_vals))
    bound_circuit = PARAM_CIRCUIT.assign_parameters(param_dict)
    return FidelityStatevectorKernel(feature_map=bound_circuit)

def centered_kta(K, y):
    n = len(y)
    H = np.eye(n) - np.ones((n, n)) / n
    K_c = H @ K @ H
    Y = np.outer(y, y)
    Y_c = H @ Y @ H

    inner = np.sum(K_c * Y_c)
    norm_K = np.linalg.norm(K_c, 'fro')
    norm_Y = np.linalg.norm(Y_c, 'fro')

    if norm_K == 0 or norm_Y == 0:
        return 0.0
    return inner / (norm_K * norm_Y)

# ------------------------- Experiment loop ---------------------------------
def run_experiment(X_raw, y, n_reps=N_REPS, verbose=True):
    results = {
        "train": {"acc": [], "kappa": [], "f1": []},
        "test": {"acc": [], "kappa": [], "f1": []},
    }

    for rep in range(n_reps):
        X_tr_full, X_te_full, y_tr, y_te = train_test_split(
            X_raw, y,
            train_size=N_TRAIN, test_size=N_TEST,
            stratify=y, random_state=rep,
        )

        pca = PCA(n_components=N_DIM, random_state=rep)
        X_tr_pca = pca.fit_transform(X_tr_full)
        X_te_pca = pca.transform(X_te_full)

        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_tr = scaler.fit_transform(X_tr_pca)
        X_te = scaler.transform(X_te_pca)

        # ---- PHASE 1: QKT Optimization (Centered KTA) ----
        def qkt_objective(theta):
            kernel = get_bound_kernel(theta)
            K_train = kernel.evaluate(x_vec=X_tr)
            return -centered_kta(K_train, y_tr)

        # Initialize the 6 dedicated parameters
        initial_theta = np.random.uniform(-np.pi, np.pi, size=6)

        res = minimize(qkt_objective, initial_theta, method='COBYLA', options={'maxiter': QKT_MAXITER})
        best_theta = res.x

        opt_kernel = get_bound_kernel(best_theta)

        def scaled_kernel_opt(X1, X2, lam):
            return opt_kernel.evaluate(x_vec=X1 * lam, y_vec=X2 * lam)

        # ---- PHASE 2: Grid search over (lambda, C) via CV on train ----
        best_score, best_C, best_lam = -1.0, None, None
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=rep)

        for lam in LAMBDA_GRID:
            K_full = scaled_kernel_opt(X_tr, X_tr, lam)
            for C in C_GRID:
                fold_acc = []
                for tr_idx, val_idx in skf.split(X_tr, y_tr):
                    K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                    K_val = K_full[np.ix_(val_idx, tr_idx)]
                    clf = SVC(kernel="precomputed", C=C)
                    clf.fit(K_tr, y_tr[tr_idx])
                    pred = clf.predict(K_val)
                    fold_acc.append(accuracy_score(y_tr[val_idx], pred))
                mean_acc = float(np.mean(fold_acc))
                if mean_acc > best_score:
                    best_score, best_C, best_lam = mean_acc, C, lam

        # ---- PHASE 3: Refit on full train split with winning hyperparams ----
        K_train = scaled_kernel_opt(X_tr, X_tr, best_lam)
        K_test = scaled_kernel_opt(X_te, X_tr, best_lam)

        clf = SVC(kernel="precomputed", C=best_C)
        clf.fit(K_train, y_tr)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        results["train"]["acc"].append(accuracy_score(y_tr, pred_train))
        results["train"]["kappa"].append(cohen_kappa_score(y_tr, pred_train))
        results["train"]["f1"].append(f1_score(y_tr, pred_train, average="macro"))

        results["test"]["acc"].append(accuracy_score(y_te, pred_test))
        results["test"]["kappa"].append(cohen_kappa_score(y_te, pred_test))
        results["test"]["f1"].append(f1_score(y_te, pred_test, average="macro"))

        if verbose:
            print(f"[rep {rep + 1:2d}/{n_reps}] KTA={-res.fun:.3f} best_C={best_C:<6} "
                  f"best_lambda={best_lam:<6} test_acc={results['test']['acc'][-1]:.3f}")

    return results

def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"

def print_table2_row(results, dataset="MNIST-1D-PCA-4", mapping="CovariantMap-Dedicated"):
    print(f"\n=== Table 2 style summary: {dataset} | Covariant QKT (n={len(results['test']['acc'])}) ===")
    header = f"{'Dataset':<18}{'Map':<26}{'Params':<10}{'Metric':<16}{'Train':<18}{'Test':<18}"
    print(header)
    print("-" * len(header))
    for metric, label in [("acc", "Accuracy"), ("kappa", "Cohen's kappa"), ("f1", "Macro F1")]:
        print(f"{dataset:<18}{mapping:<26}{'6':<10}{label:<16}"
              f"{fmt(results['train'][metric]):<18}{fmt(results['test'][metric]):<18}")

if __name__ == "__main__":
    X_raw, y = load_mnist1d_binary()
    print(f"Loaded MNIST-1D digits {DIGITS}: {X_raw.shape[0]} samples, raw dim={X_raw.shape[1]}")
    results = run_experiment(X_raw, y, n_reps=N_REPS)
    print_table2_row(results)

Loaded MNIST-1D digits (3, 5): 1000 samples, raw dim=40
[rep  1/30] KTA=0.429 best_C=1      best_lambda=1.0    test_acc=0.908
[rep  2/30] KTA=0.405 best_C=1      best_lambda=0.5    test_acc=0.928
[rep  3/30] KTA=0.272 best_C=10     best_lambda=0.5    test_acc=0.856
[rep  4/30] KTA=0.368 best_C=10     best_lambda=0.5    test_acc=0.876
[rep  5/30] KTA=0.375 best_C=100    best_lambda=0.5    test_acc=0.888
[rep  6/30] KTA=0.271 best_C=1      best_lambda=0.5    test_acc=0.884
[rep  7/30] KTA=0.347 best_C=1      best_lambda=0.5    test_acc=0.888
[rep  8/30] KTA=0.434 best_C=10     best_lambda=0.5    test_acc=0.868
[rep  9/30] KTA=0.336 best_C=0.1    best_lambda=0.5    test_acc=0.864
[rep 10/30] KTA=0.415 best_C=10     best_lambda=0.5    test_acc=0.900
[rep 11/30] KTA=0.299 best_C=10     best_lambda=0.5    test_acc=0.880
[rep 12/30] KTA=0.422 best_C=100    best_lambda=0.5    test_acc=0.896
[rep 13/30] KTA=0.354 best_C=10     best_lambda=0.5    test_acc=0.844
[rep 14/30] KTA=0.482 best_C=100  